# Day 4 · numpy 核心

**上午**：读 numpy 官方 Quickstart
**下午**：数组创建/索引 + 手写矩阵乘法 + 广播实验
**晚上**：用 numpy 重写前两天的练习

> 核心思想：**向量化代替 for 循环**。这是后面所有 PyTorch 代码的思维方式。

## 1. 创建数组与基本属性

In [1]:
import numpy as np
a = np.array([[1,2,3],[4,5,6]])
print(a.shape, a.ndim, a.dtype)
print(np.zeros((2,3)), np.ones(4), np.arange(0,10,2))
print(np.linspace(0, 1, 5))
print(np.random.randn(2,3))

(2, 3) 2 int32
[[0. 0. 0.]
 [0. 0. 0.]] [1. 1. 1. 1.] [0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[[ 0.4815094  -1.00598952 -0.81562235]
 [ 0.86857037  1.12672012 -1.27784095]]


## 2. 索引、切片、布尔索引

In [2]:
a = np.arange(12).reshape(3,4)
print(a[1, 2], a[:, 1], a[::2, -1])
print(a[a > 6])
a[a > 6] = 0
print(a)

6 [1 5 9] [ 3 11]
[ 7  8  9 10 11]
[[0 1 2 3]
 [4 5 6 0]
 [0 0 0 0]]


## 3. 广播：亲手验证规则

In [ ]:
a = np.ones((3,4))
b = np.array([1,2,3,4])
print((a + b).shape)  # 规则：从尾部对齐，维度相同或一方为 1 才能广播
# 下面哪个会报错？先猜再运行：
# c = np.ones((3,5)); c + b 会报错
# d = np.ones((1,4)); d + b 不会报错

(3, 4)


## 4. 下午：手写矩阵乘法并对比速度

In [7]:
# import time
# A = np.random.rand(200, 200); B = np.random.rand(200, 200)
# t = time.time(); C1 = A @ B; print('np.matmul:', time.time()-t, '秒')
# 用三重 for 循环实现 C2，再 np.allclose(C1, C2) 验证
# 然后换 1000x1000 只跑 np.matmul，感受差距

import time
import numpy as np

# ---------- 200x200：NumPy 版 ----------
A = np.random.rand(200, 200)
B = np.random.rand(200, 200)

t = time.time()
C1 = A @ B
t_numpy = time.time() - t
print(f'np.matmul (200x200): {t_numpy:.4f} 秒')

# ---------- 200x200：纯 Python 三重循环 ----------
def matmul_python(A, B):
    n, m = A.shape
    m2, p = B.shape
    assert m == m2, '维度不匹配'
    C = [[0.0] * p for _ in range(n)]
    for i in range(n):
        for j in range(p):
            s = 0.0
            for k in range(m):
                s += A[i, k] * B[k, j]
            C[i][j] = s
    return C

t = time.time()
C2 = matmul_python(A, B)
t_python = time.time() - t
print(f'三重循环  (200x200): {t_python:.4f} 秒')

# ---------- 验证两种结果一致 ----------
print('np.allclose(C1, C2):', np.allclose(C1, np.array(C2)))
print(f'NumPy 快了约 {t_python / t_numpy:.0f} 倍')

# ---------- 1000x1000：只跑 NumPy ----------
A = np.random.rand(1000, 1000)
B = np.random.rand(1000, 1000)

t = time.time()
C3 = A @ B
t_numpy_1000 = time.time() - t
print(f'np.matmul (1000x1000): {t_numpy_1000:.4f} 秒')

np.matmul (200x200): 0.0016 秒
三重循环  (200x200): 2.2597 秒
np.allclose(C1, C2): True
NumPy 快了约 1392 倍
np.matmul (1000x1000): 0.0249 秒


## 5. 练习：不用 for 循环

In [11]:
# 练习 1：X 是 (100,3) 随机矩阵，每行减去该行的均值
# X = np.random.rand(100,3)
import numpy as np

X = np.random.rand(100, 3)          # shape (100, 3)

row_mean = X.mean(axis=1)           # 每行均值，shape (100,)
X1 = X - row_mean[:, None]          # 扩成 (100,1)，广播到 (100,3)
print(X1.shape)                     # (100, 3)
print(np.allclose(X1.mean(axis=1), 0))  # True，每行均值变成 0
# 练习 2：每列做标准化（减均值除标准差）
col_mean = X.mean(axis=0)           # 每列均值，shape (3,)
col_std = X.std(axis=0)             # 每列标准差，shape (3,)
X2 = (X - col_mean) / col_std       # 广播：(100,3) - (3,) → (100,3)
print(X2.shape)                     # (100, 3)
print(np.allclose(X2.mean(axis=0), 0))   # True，每列均值约 0
print(np.allclose(X2.std(axis=0), 1))    # True，每列标准差约 1
# 练习 3：one-hot：labels = np.array([0,2,1,0]) -> np.eye(3)[labels]
labels = np.array([0, 2, 1, 0])
onehot = np.eye(3)[labels]
print(onehot)
# [[1. 0. 0.]
#  [0. 0. 1.]
#  [0. 1. 0.]
#  [1. 0. 0.]]
print(onehot.shape)   # (4, 3)
# 练习 4：对 10000 个正态样本求平方和、均值、标准差
samples = np.random.randn(10000)    # 标准正态，均值 0，标准差 1

sq_sum = (samples ** 2).sum()       # 平方和
mean = samples.mean()               # 均值
std = samples.std()                 # 标准差（总体标准差，ddof=0）

print('平方和:', sq_sum)
print('均值:', mean)
print('标准差:', std)

(100, 3)
True
(100, 3)
True
True
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]
(4, 3)
平方和: 9837.157100022665
均值: -0.00927746283192657
标准差: 0.991781043721682


## 6. 晚间：用 numpy 重写 Day2 的词频统计

In [12]:
# 提示：np.unique(text.split(), return_counts=True)
import numpy as np

text = 'the quick brown fox jumps over the lazy dog the fox runs'

words = np.array(text.split())                    # 转成 numpy 数组
unique, counts = np.unique(words, return_counts=True)

print(unique)
# ['brown' 'dog' 'fox' 'jumps' 'lazy' 'over' 'quick' 'runs' 'the']

print(counts)
# [1 1 2 1 1 1 1 1 3]

# 合并成「词 → 次数」的字典
freq = dict(zip(unique, counts))
print(freq)
# {'brown': 1, 'dog': 1, 'fox': 2, 'jumps': 1, 'lazy': 1, 'over': 1, 'quick': 1, 'runs': 1, 'the': 3}

['brown' 'dog' 'fox' 'jumps' 'lazy' 'over' 'quick' 'runs' 'the']
[1 1 2 1 1 1 1 1 3]
{'brown': 1, 'dog': 1, 'fox': 2, 'jumps': 1, 'lazy': 1, 'over': 1, 'quick': 1, 'runs': 1, 'the': 3}
